## Data Splitting

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/cleaned_resale_transactions_v2.csv")
df.head()

,month,flat_type,storey_range,floor_area_sqm,lease_commence_date,resale_price,town_name,flatm_name,year,remaining_lease_months,distance_to_school,distance_to_mrt,distance_to_mall
0,5,5 ROOM,8.0,123.0,1989,488000.0,PASIR RIS,Improved,2018,843,246.038467,639.602712,694.155720
1,10,4 ROOM,2.0,100.0,1999,345000.0,SENGKANG,Model A,2016,972,181.817361,282.929658,728.841888
2,10,5 ROOM,20.0,110.0,2002,370000.0,CHOA CHU KANG,Improved,2018,990,434.160991,796.596944,727.758783
3,12,4 ROOM,14.0,103.0,1984,432000.0,HOUGANG,Model A,2015,804,156.233044,984.042903,886.398752
4,9,5 ROOM,8.0,122.0,1992,500000.0,SERANGOON,Improved,2017,890,188.343937,1938.596051,557.617856


In [3]:
# Separate the data into features and target
X = df.drop(columns='resale_price')
y = df['resale_price']

In [4]:
# Show X - Note the uppercase
X

,month,flat_type,storey_range,floor_area_sqm,lease_commence_date,town_name,flatm_name,year,remaining_lease_months,distance_to_school,distance_to_mrt,distance_to_mall
0,5,5 ROOM,8.0,123.0,1989,PASIR RIS,Improved,2018,843,246.038467,639.602712,694.155720
1,10,4 ROOM,2.0,100.0,1999,SENGKANG,Model A,2016,972,181.817361,282.929658,728.841888
2,10,5 ROOM,20.0,110.0,2002,CHOA CHU KANG,Improved,2018,990,434.160991,796.596944,727.758783
3,12,4 ROOM,14.0,103.0,1984,HOUGANG,Model A,2015,804,156.233044,984.042903,886.398752
4,9,5 ROOM,8.0,122.0,1992,SERANGOON,Improved,2017,890,188.343937,1938.596051,557.617856
...,...,...,...,...,...,...,...,...,...,...,...,...
84460,9,5 ROOM,11.0,111.0,2001,SEMBAWANG,Improved,2015,1020,360.241675,90.971298,186.046578
84461,6,5 ROOM,8.0,110.0,2002,ANG MO KIO,Improved,2015,1032,133.365490,317.871018,435.717120
84462,9,5 ROOM,2.0,110.0,2003,HOUGANG,Improved,2017,1018,309.310818,1336.889395,422.116607
84463,4,4 ROOM,2.0,91.0,1984,ANG MO KIO,New Generation,2016,792,555.716546,1233.238634,836.515538


In [5]:
# Show y - Note the lowercase
y

0        488000.0
1        345000.0
2        370000.0
3        432000.0
4        500000.0
           ...   
84460    468000.0
84461    760000.0
84462    380000.0
84463    406000.0
84464    425000.0
Name: resale_price, Length: 84465, dtype: float64

In [6]:
from sklearn.model_selection import train_test_split

# Split the data into training (80%) and test-validation (20%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Split the test-validation set (20%) into validation (10%) and test (10%) sets
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [7]:
# Display the shapes of the splits to verify
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

Training set shape: (67572, 12) (67572,)
Validation set shape: (8446, 12) (8446,)
Test set shape: (8447, 12) (8447,)


## Feature Scaling

Before deciding whether to normalise or standardise the numerical features, I looked at their distributions during the preliminary data exploration. I found that most of the features were nearly normal distributed but differed significantly in scale. Based on this, I decided that standardisation would be more appropriate for my analysis because:

- Distribution: The numerical features are approximately normally distributed.
- Scales: The features have very different ranges, which could cause some variables to have a greater influence than others.
- Algorithm Assumptions: Standardisation is suitable for the machine learning algorithms I plan to use later, particularly linear models and other scale-sensitive algorithms.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Define numerical features to be standardized
numerical_features = ['floor_area_sqm', 'remaining_lease_months', 'lease_commence_date', 'year',
                      'distance_to_school', 'distance_to_mrt', 'distance_to_mall']

# Create a numerical transformer pipeline
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

## Feature Encoding

In [9]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Define nomial features to be one-hot encoded
nominal_features = ['month', 'town_name', 'flatm_name']

# Define ordinal features to be ordinally encoded
ordinal_features = ['flat_type']

# Define the ordinal categories for flat_type
flat_type_categories = ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'MULTI-GENERATION', 'EXECUTIVE']

# Define passthrough features that will not be transformed
passthrough_features = ['storey_range']

# Create a nominal transformer pipeline
nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create an ordinal transformer pipeline
ordinal_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[flat_type_categories], handle_unknown='use_encoded_value', unknown_value=-1))  
])

In [10]:
from sklearn.compose import ColumnTransformer

# Combine transformers into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('nom', nominal_transformer, nominal_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('pass', 'passthrough', passthrough_features) # Pass through the storey_range feature without transformation
    ],
    remainder='passthrough',
    n_jobs=-1
    )

In [11]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feat

## Multivariate Linear Regression

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

# Create the pipeline with a linear regression model
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Fit the pipeline to the training data
lr_pipeline.fit(X_train, y_train)

# Predict on the validation set with Linear Regression
y_val_pred_lr = lr_pipeline.predict(X_val)

# Calculate regression metrics for validation set with Linear Regression
val_mae_lr = mean_absolute_error(y_val, y_val_pred_lr)
val_mse_lr = mean_squared_error(y_val, y_val_pred_lr)
val_rmse_lr = root_mean_squared_error(y_val, y_val_pred_lr) # RMSE is the square root of MSE
val_r2_lr = r2_score(y_val, y_val_pred_lr)

# Display the metrics for Linear Regression
print("Linear Regression Validation Metrics:")
print(f"Linear Regression MAE: {val_mae_lr}")
print(f"Linear Regression MSE: {val_mse_lr}")
print(f"Linear Regression RMSE: {val_rmse_lr}")
print(f"Linear Regression R²: {val_r2_lr}")

Linear Regression Validation Metrics:
Linear Regression MAE: 40214.71598524548
Linear Regression MSE: 2751618970.582802
Linear Regression RMSE: 52455.87641611569
Linear Regression R²: 0.8713845459976413


## Ridge Regression
This Ridge Regression model uses the default `alpha = 1`. It is trained on the preprocessed features via the pipeline and evaluated on the validation set using MAE, MSE, RMSE, and R².

In [18]:
from sklearn.linear_model import Ridge

# Fit Ridge Regression with lambda (alpha in scikit-learn)
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1))
])
ridge_pipeline.fit(X_train, y_train)

# Predict on the validation set with Ridge Regression
y_val_pred_ridge = ridge_pipeline.predict(X_val)

# Calculate regression metrics for validation set with Ridge Regression
val_mae_ridge = mean_absolute_error(y_val, y_val_pred_ridge)
val_mse_ridge = mean_squared_error(y_val, y_val_pred_ridge)
val_rmse_ridge = root_mean_squared_error(y_val, y_val_pred_ridge)  # RMSE is the square root of MSE
val_r2_ridge = r2_score(y_val, y_val_pred_ridge)

# Display the metrics for Ridge Regression
print("Ridge Regression Metrics:")
print(f"Ridge Validation MAE: {val_mae_ridge}")
print(f"Ridge Validation MSE: {val_mse_ridge}")
print(f"Ridge Validation RMSE: {val_rmse_ridge}")
print(f"Ridge Validation R²: {val_r2_ridge}")

Ridge Regression Metrics:
Ridge Validation MAE: 40215.33088714012
Ridge Validation MSE: 2751140457.275805
Ridge Validation RMSE: 52451.31511483582
Ridge Validation R²: 0.871406912541441


## Lasso Regression
This Lasso Regression model uses the default `alpha = 1`. It is trained on the preprocessed features via the pipeline and evaluated on the validation set using MAE, MSE, RMSE, and R².

In [17]:
from sklearn.linear_model import Lasso

# Fit Lasso Regression with default alpha
lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Lasso(alpha=1))
])
lasso_pipeline.fit(X_train, y_train)

# Predict on the validation set with Lasso Regression
y_val_pred_lasso = lasso_pipeline.predict(X_val)

# Calculate regression metrics for validation set with Lasso Regression
val_mae_lasso = mean_absolute_error(y_val, y_val_pred_lasso)
val_mse_lasso = mean_squared_error(y_val, y_val_pred_lasso)
val_rmse_lasso = root_mean_squared_error(y_val, y_val_pred_lasso)  # RMSE is the square root of MSE
val_r2_lasso = r2_score(y_val, y_val_pred_lasso)

# Display the metrics for Lasso Regression
print("Lasso Regression Metrics:")
print(f"Lasso Validation MAE: {val_mae_lasso}")
print(f"Lasso Validation MSE: {val_mse_lasso}")
print(f"Lasso Validation RMSE: {val_rmse_lasso}")
print(f"Lasso Validation R²: {val_r2_lasso}")

Lasso Regression Metrics:
Lasso Validation MAE: 40215.274919968106
Lasso Validation MSE: 2751475942.182147
Lasso Validation RMSE: 52454.51307735252
Lasso Validation R²: 0.8713912313937234


/opt/anaconda3/envs/my-new-env/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.576579e+13, tolerance: 1.457e+11
  model = cd_fast.sparse_enet_coordinate_descent(


## Ridge Regression
This Ridge Regression model uses `GridSearchCV` with 5-fold cross-validation to search for the optimal combination of `alpha` (0.1, 1, 10, 100, 1000) and `fit_intercept` (True, False), using R² as the scoring metric. The best-performing combination is then evaluated on the validation set using MAE, MSE, RMSE, and R².

In [21]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'regressor__alpha': [0.1, 1, 10, 100, 1000],
    'regressor__fit_intercept': [True, False]
}

# Create the pipeline with a ridge regression model
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

# Perform grid search with cross-validation for Ridge regression
ridge_grid_search = GridSearchCV(ridge_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
ridge_grid_search.fit(X_train, y_train)

# Print the best parameters found by grid search for Ridge regression
print("Best Ridge parameters:", ridge_grid_search.best_params_)
best_alpha_ridge = ridge_grid_search.best_params_['regressor__alpha']

# Evaluate the best Ridge model on the validation set
best_ridge_model = ridge_grid_search.best_estimator_
y_val_pred_ridge = best_ridge_model.predict(X_val)

# Calculate regression metrics for validation set for Ridge
val_mae_ridge = mean_absolute_error(y_val, y_val_pred_ridge)
val_mse_ridge = mean_squared_error(y_val, y_val_pred_ridge)
val_rmse_ridge = root_mean_squared_error(y_val, y_val_pred_ridge)  # RMSE is the square root of MSE
val_r2_ridge = r2_score(y_val, y_val_pred_ridge)

print(f"Best Ridge Regression Metrics (Tuned alpha={best_alpha_ridge})")
print(f"Ridge Validation MAE: {val_mae_ridge}")
print(f"Ridge Validation MSE: {val_mse_ridge}")
print(f"Ridge Validation RMSE: {val_rmse_ridge}")
print(f"Ridge Validation R²: {val_r2_ridge}")

Best Ridge parameters: {'regressor__alpha': 0.1, 'regressor__fit_intercept': True}
Best Ridge Regression Metrics (Tuned alpha=0.1)
Ridge Validation MAE: 40218.30052924841
Ridge Validation MSE: 2751040442.377142
Ridge Validation RMSE: 52450.36169920224
Ridge Validation R²: 0.8714115874116668


## Lasso Regression
This Lasso Regression model uses `GridSearchCV` with 5-fold cross-validation to search for the optimal combination of `alpha` (0.1, 1, 10, 100, 1000) and `fit_intercept` (True, False), using R² as the scoring metric. The best-performing combination is then evaluated on the validation set using MAE, MSE, RMSE, and R².

In [22]:
param_grid = {
    'regressor__alpha': [0.1, 1, 10, 100, 1000],
    'regressor__fit_intercept': [True, False]
}

# Create the pipeline with a lasso regression model
lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Lasso())
])

# Perform grid search with cross-validation for Lasso regression
lasso_grid_search = GridSearchCV(lasso_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
lasso_grid_search.fit(X_train, y_train)

# Print the best parameters found by grid search for Lasso regression
print("Best Lasso parameters:", lasso_grid_search.best_params_)
best_alpha_lasso = lasso_grid_search.best_params_['regressor__alpha']

# Evaluate the best Lasso model on the validation set
best_lasso_model = lasso_grid_search.best_estimator_
y_val_pred_lasso = best_lasso_model.predict(X_val)

# Calculate regression metrics for validation set for Lasso
val_mae_lasso = mean_absolute_error(y_val, y_val_pred_lasso)
val_mse_lasso = mean_squared_error(y_val, y_val_pred_lasso)
val_rmse_lasso = root_mean_squared_error(y_val, y_val_pred_lasso)  # RMSE is the square root of MSE
val_r2_lasso = r2_score(y_val, y_val_pred_lasso)

print(f"Best Lasso Regression Metrics (Tuned alpha={best_alpha_lasso}):")
print(f"Lasso Validation MAE: {val_mae_lasso}")
print(f"Lasso Validation MSE: {val_mse_lasso}")
print(f"Lasso Validation RMSE: {val_rmse_lasso}")  
print(f"Lasso Validation R²: {val_r2_lasso}")

/opt/anaconda3/envs/my-new-env/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.833310e+13, tolerance: 1.158e+12
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/anaconda3/envs/my-new-env/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.430282e+13, tolerance: 1.161e+12
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/anaconda3/envs/my-new-env/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features 

Best Lasso parameters: {'regressor__alpha': 0.1, 'regressor__fit_intercept': False}
Best Lasso Regression Metrics (Tuned alpha=0.1):
Lasso Validation MAE: 40216.56008881131
Lasso Validation MSE: 2751727517.0735836
Lasso Validation RMSE: 52456.91105158198
Lasso Validation R²: 0.8713794723459688


/opt/anaconda3/envs/my-new-env/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.246500e+13, tolerance: 1.448e+12
  model = cd_fast.sparse_enet_coordinate_descent(


| Model | MAE | MSE | RMSE | R² |
|---|---|---|---|---|
| Best Ridge Regression (Tuned alpha=0.1) | 40218.30052924841 | 2751040442.377142 | 52450.36169920224 | 0.8714115874116668 |
| Ridge Regression (Default alpha=1) | 40215.33088714012 | 2751140457.275805 | 52451.31511483582 | 0.871406912541441 |
| Best Lasso Regression (Tuned alpha=0.1) | 40216.56008881131 | 2751727517.0735836 | 52456.91105158198 | 0.8713794723459688 |
| Lasso Regression (Default alpha=1) | 40215.274919968106 | 2751475942.182147 | 52454.51307735252 | 0.8713912313937234 |
| Linear Regression | 40214.71598524548 | 2751618970.582802 | 52455.87641611569 | 0.8713845459976413 |

After evaluating the metrics, it is evident that the best **Ridge Regression** model with tuned hyperparameters of `alpha=0.1` performs the best overall. It achieved the lowest MSE and RMSE, along with the highest R², indicating that it made the most accurate predictions and explains the most variance in the HDB resale prices.

Therefore, the best ridge regression model will be the chosen model for predicting HDB resale prices. This model offers a good balance of low error rates and high explanatory power, making it the most reliable choice for our task.

## Final Model Evaluation
The best performing model - Ridge Regression with tuned hyperparameters of `alpha=0.1` will be used on the test set to see how well it performs on completely unseen data.

In [23]:
# Best model from Ridge regression (Grid Search)
best_ridge_model = ridge_grid_search.best_estimator_

# Predict on the test set with Ridge Regression
y_test_pred_ridge = best_ridge_model.predict(X_test)

# Calculate regression metrics for the test set for Ridge
test_mae_ridge = mean_absolute_error(y_test, y_test_pred_ridge)
test_mse_ridge = mean_squared_error(y_test, y_test_pred_ridge)
test_rmse_ridge = root_mean_squared_error(y_test, y_test_pred_ridge)
test_r2_ridge = r2_score(y_test, y_test_pred_ridge)

print("Best Ridge Regression Model, Final Test Metrics:")
print(f"Final Test MAE: {test_mae_ridge}")
print(f"Final Test MSE: {test_mse_ridge}")
print(f"Final Test RMSE: {test_rmse_ridge}")
print(f"Final Test R²: {test_r2_ridge}")

Best Ridge Regression Model, Final Test Metrics:
Final Test MAE: 39382.07840306421
Final Test MSE: 2646397060.3420706
Final Test RMSE: 51443.14395856916
Final Test R²: 0.877247593334062


The model performed slightly better on the test set than on the validation set across all four metrics (MAE, MSE, RMSE, and R²). This suggests that the Ridge Regression model is able to generalise well to unseen data rather than only performing well on the data used during model development.

Overall, the consistent performance across both datasets gives me confidence that the model is reasonably robust and reliable for predicting HDB resale prices.